In [1]:
import pandas as pd

files = [
    "andhra_rainfall_statistics.csv",
    "fertilizerstudy.csv",
    "crop_production.csv",
    "GW_Levels_1776197307778.csv",
    "gwq_chemical_parameter.csv",
    "Soil moisture summary_1776197357611.csv",
    "ap_district_population.csv"
]

for f in files:
    try:
        df = pd.read_csv("data/" + f)
        print("\nFILE:", f)
        print(df.columns.tolist())
        print(df.head(2))
    except Exception as e:
        print("\nFILE:", f, "ERROR:", e)


FILE: andhra_rainfall_statistics.csv
['level', 'name', 'period', 'mean_rainfall_mm', 'cv_percent']
   level            name period  mean_rainfall_mm  cv_percent
0  State  Andhra Pradesh   June              96.3        55.4
1  State  Andhra Pradesh   July             127.5        37.7

FILE: fertilizerstudy.csv
['S.No', 'District', 'Total Requirement', 'Unnamed: 3', 'Unnamed: 4', 'Kharif', 'Unnamed: 6', 'Unnamed: 7', 'Rabi', 'Unnamed: 9', 'Unnamed: 10']
  S.No    District Total Requirement Unnamed: 3 Unnamed: 4     Kharif  \
0  NaN         NaN                 N          P          K          N   
1    1  Srikakulam         40,295.35  22,101.25  23,818.18  30,650.50   

  Unnamed: 6 Unnamed: 7      Rabi Unnamed: 9 Unnamed: 10  
0          P          K         N          P           K  
1  18,102.30  18,992.61  9,644.85   3,998.95    4,825.58  

FILE: crop_production.csv
['S.No', 'District', 'Production and Price Forecast for RABI 2018-19*', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unn

In [2]:
pop = pd.read_csv("data/ap_district_population.csv")

pop = pop[pop["level"]=="District"].copy()
pop = pop.rename(columns={"name":"District"})

pop = pop[["District","population_2011","growth_2001_2011_pct"]]

In [3]:
chem = pd.read_csv("data/gwq_chemical_parameter.csv")

chem_cols = [
    "District",
    "Latitude",
    "Longitude",
    "Potential of Hydrogen (pH)",
    "Electric Conductivity (μS/cm)",
    "Total Dissolved Solids (mg/L)",
    "Fluoride (mg/L)"
]

chem = chem[chem_cols].copy()

for c in chem.columns[1:]:
    chem[c] = pd.to_numeric(chem[c], errors="coerce")

chem = chem.groupby("District").mean(numeric_only=True).reset_index()

In [4]:
fert = pd.read_csv("data/fertilizerstudy.csv")

fert = fert.rename(columns={"District":"District"})
fert["Total Requirement"] = pd.to_numeric(fert["Total Requirement"], errors="coerce")

fert = fert.groupby("District")["Total Requirement"].mean().reset_index()

In [5]:
master = pop.merge(chem, on="District", how="outer")
master = master.merge(fert, on="District", how="outer")

In [6]:
master["District"] = master["District"].str.upper().str.strip()
master = master.groupby("District").mean(numeric_only=True).reset_index()

In [7]:
print(master.shape)
print(master.head())

(18, 10)
        District  population_2011  growth_2001_2011_pct   Latitude  Longitude  \
0      ANANTAPUR        4081148.0                 12.10  19.921325  76.404747   
1  ANANTHAPURAMU              NaN                   NaN        NaN        NaN   
2       CHITTOOR        4174064.0                 11.43  13.450400  79.164123   
3       CUDDAPAH              NaN                   NaN  14.468357  78.702739   
4  EAST GODAVARI        5154296.0                  5.16  17.036278  82.065526   

   Potential of Hydrogen (pH)  Electric Conductivity (μS/cm)  \
0                    7.933620                    1541.424650   
1                         NaN                            NaN   
2                    7.782504                    1470.003454   
3                    7.860798                    1880.119403   
4                    8.000748                    1613.055819   

   Total Dissolved Solids (mg/L)  Fluoride (mg/L)  Total Requirement  
0                     647.772424              Na

In [8]:
def clean_name(x):
    if pd.isna(x):
        return x
    x = str(x).upper().strip()

    mapping = {
        "ANANTHAPURAMU": "ANANTAPUR",
        "CUDDAPAH": "KADAPA",
        "YSR KADAPA": "KADAPA",
        "Y.S.R KADAPA": "KADAPA",
        "SPSR NELLORE": "NELLORE",
        "SRI POTTI SRIRAMULU NELLORE": "NELLORE",
        "VISHAKHAPATNAM": "VISAKHAPATNAM",
        "EAST GODAVARI ": "EAST GODAVARI",
        "WEST GODAVARI ": "WEST GODAVARI"
    }

    return mapping.get(x, x)

In [9]:
pop["District"] = pop["District"].apply(clean_name)
chem["District"] = chem["District"].apply(clean_name)
fert["District"] = fert["District"].apply(clean_name)

In [10]:
master = pop.merge(chem, on="District", how="outer")
master = master.merge(fert, on="District", how="outer")

print(master.shape)
print(master["District"])

(15, 10)
0         ANANTAPUR
1          CHITTOOR
2     EAST GODAVARI
3            GUNTUR
4            KADAPA
5           KHAMMAM
6           KRISHNA
7           KURNOOL
8           NELLORE
9          PRAKASAM
10       RANGAREDDY
11       SRIKAKULAM
12    VISAKHAPATNAM
13     VIZIANAGARAM
14    WEST GODAVARI
Name: District, dtype: str


In [15]:
chem.shape
chem.head()
chem.columns

Index(['District', 'Latitude', 'Longitude', 'Potential of Hydrogen (pH)',
       'Electric Conductivity (μS/cm)', 'Total Dissolved Solids (mg/L)',
       'Fluoride (mg/L)'],
      dtype='str')